In [0]:
from pyspark.sql.functions import col, concat_ws, regexp_replace, split, upper, current_date
import os
import sys
from typing import List
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import importlib
import utils.custom_utils
importlib.reload(utils.custom_utils)
from utils.custom_utils import transformations

In [0]:
current_dir = os.getcwd()
print(current_dir)
sys.path.append(current_dir)

/Workspace/Users/pragathimails@gmail.com/pyspark_dbt_project


In [0]:
def transform_customers(df):
    df = df.withColumn('domain', split(col('email'), '@')[1])
    df = df.withColumn('phone_number', regexp_replace('phone_number', r'[^0-9]', ''))
    df = df.withColumn('full_name', concat_ws(' ', col('first_name'), col('last_name')))
    df = df.drop('first_name', 'last_name')
    return df

In [0]:
def transform_vehicles(df):
    df = df.withColumn("make", upper(col("make")))
    return df
    

In [0]:
def transform_trips(df):
    df = df.drop("start_location","end_location","payment_method")
    return df

In [0]:
def transform_passthrough(df):
    return df

In [0]:
def transform_drivers(df):
    df = df.withColumn("phone_number", regexp_replace("phone_number", r'[^0-9]', ""))
    df = df.withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name")))
    df = df.drop("first_name", "last_name")
    return df

In [0]:
customers_good = (
    (col("signup_date") <= current_date())
)

drivers_good = (
    ((col("driver_rating") >= 0) & (col("driver_rating") <= 5))
)

locations_good = (
    (col("latitude").between(-90, 90)) &
    (col("longitude").between(-180, 180))
)

vehicles_good = (
    (col("year").between(1990, 2027)) &
    (col("vehicle_type").isin("Hatchback","Sedan","SUV","Van","Luxury"))
)

payments_good = (
    (col("payment_method").isin("Cash", "Card", "Wallet")) &
    (col("payment_status").isin("Pending", "Success","Failed")) &
    (col("amount") > 0)
)

In [0]:

trips_good = (
    (col("distance_km") > 0) &
    (col("fare_amount") > 0) &
    (col("trip_status").isin("Cancelled", "Ongoing", "Completed")) &
    (col("trip_start_time") < col("trip_end_time"))
)

In [0]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

obj = transformations()

def process_table(table, key, custom_transform, good_condition, reason):
    try:
        logger.info(f"Silver processing started: {table}")

        # Reading bronze
        df = spark.read.table(f"pysparkdbt.bronze.{table}")

        # custom ingestion
        df = custom_transform(df)

        # common
        df = obj.trim_strings(df)
        df = obj.handle_nulls(df, key)
        df = obj.dedup(df, [key], "last_updated_timestamp")
        df = obj.transform_timestamp(df)
        df = obj.validate_and_quarantine(df, good_condition, table, reason)

        # upsert
        if not spark.catalog.tableExists(f"pysparkdbt.silver.{table}"):
            df.write.format("delta").mode("append").saveAsTable(f"pysparkdbt.silver.{table}")
        else:
            obj.upsert(spark, df, [key], table, "last_updated_timestamp")

        logger.info(f"Silver processing completed: {table}")

    except Exception as e:
        logger.error(f"Silver processing failed for {table}: {e}")
        raise

In [0]:
tables_config = [
    ("customers", "customer_id", transform_customers, customers_good,"customers DQ"),
    ("drivers",   "driver_id",   transform_drivers, drivers_good,"drivers DQ"),
    ("locations", "location_id", transform_passthrough, locations_good,"locations DQ"),
    ("vehicles",  "vehicle_id",  transform_vehicles, vehicles_good,"vehicles DQ"),
    ("trips",     "trip_id",     transform_trips, trips_good,"trips DQ"),
    ("payments",  "payment_id",  transform_passthrough, payments_good,"payments DQ"),
]

for table, key, custom_transform, good_condition, reason in tables_config:
    process_table(table, key, custom_transform, good_condition, reason)

2026-08-13 06:46:58,659 [INFO] Silver processing started: customers
2026-08-13 06:46:58,664 [INFO] Trimming Strings
2026-08-13 06:46:59,013 [INFO] Trimming Strings completed
2026-08-13 06:46:59,014 [INFO] Handling nulls: customer_id
2026-08-13 06:46:59,015 [INFO] Handling nulls is completed: customer_id
2026-08-13 06:46:59,015 [INFO] Deduplication started on columns ['customer_id']
2026-08-13 06:46:59,016 [INFO] Deduplication completed on columns ['customer_id']
2026-08-13 06:46:59,016 [INFO] Transforming timestamp
2026-08-13 06:46:59,017 [INFO] Transforming timestamp completed
2026-08-13 06:46:59,018 [INFO] Validating and quarantining data
2026-08-13 06:46:59,477 [INFO] No rows to quarantine
2026-08-13 06:46:59,788 [INFO] Starting upsert into silver.customers
2026-08-13 06:47:04,943 [INFO] Upsert into silver.customers completed successfully
2026-08-13 06:47:04,944 [INFO] Silver processing completed: customers
2026-08-13 06:47:04,945 [INFO] Silver processing started: drivers
2026-08-13